# YOLOv12n x VisDrone -- `cagi` Notebook

> ACCV 2026 | Adaptive Computation for Lightweight UAV Object Detection

**Idea**: Complexity-Aware Gated Inference (CAGI)

**Mo ta**: Scene Complexity Predictor quyet dinh so C3k2 blocks: 1/2/4 theo EASY/MEDIUM/HARD.

> Thoi gian: ~3-4h tren Colab T4

---

### Huong dan
1. Runtime -> Change runtime type -> T4 GPU
2. Chay tung cell tu tren xuong duoi
3. Ket qua luu vao Drive: MyDrive/yolov12n-visdrone/runs/
4. Metrics: MyDrive/yolov12n-visdrone/metrics.csv

## 1. Kiem tra GPU

In [ ]:
import subprocess, torch
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode==0 else 'No GPU')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT = '/content/drive/MyDrive/yolov12n-visdrone'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive: {DRIVE_ROOT}')

## 3. Cai dat dependencies

In [ ]:
%pip install -q ultralytics>=8.3.0 omegaconf>=2.3.0 rich>=13.0.0 thop
import ultralytics; ultralytics.checks()
print(f'ultralytics: {ultralytics.__version__}')

## 4. Upload project code

Chon **mot trong 3 cach**:

- Cach 1: Upload file `.zip` tu may tinh
- Cach 2: Copy tu Google Drive
- Cach 3: `git clone` tu GitHub

In [ ]:
import os, shutil, zipfile
PROJECT_DIR = '/content/yolov12n-visdrone'

# -- Cach 1: Upload zip -------------------------------------------------
# from google.colab import files
# up = files.upload()
# with zipfile.ZipFile(list(up.keys())[0]) as z: z.extractall('/content/')

# -- Cach 2: Tu Drive ---------------------------------------------------
# shutil.copytree('/content/drive/MyDrive/yolov12n-visdrone-code',
#                 PROJECT_DIR, dirs_exist_ok=True)

# -- Cach 3: Git clone --------------------------------------------------
# !git clone https://github.com/your-username/yolov12n-visdrone {PROJECT_DIR}

print('OK' if os.path.exists(PROJECT_DIR) else 'Project chua duoc upload')

In [ ]:
import os
os.chdir('/content/yolov12n-visdrone')
print(os.getcwd(), os.listdir('.'))

## 5. Load VisDrone dataset tu Google Drive

> Dataset da duoc convert sang YOLO format, zip lai thanh `VisDrone.zip`
> va upload len Drive tai `MyDrive/yolov12n-visdrone/datasets/VisDrone.zip`.
> **Khong can chay `prepare_visdrone.py` lai.** Notebook se tu dong giai nen.

**Thu tu uu tien:**
1. `VisDrone.zip` tren Drive → giai nen (~60s)
2. Thu muc `VisDrone/` (da giai nen) tren Drive → symlink (~2s)
3. Khong tim thay → bao loi, kiem tra lai duong dan Drive

In [ ]:
import os, zipfile, time

PROJECT_DIR = '/content/yolov12n-visdrone'
YOLO_LOCAL  = os.path.join(PROJECT_DIR, 'datasets', 'VisDrone')
DRIVE_BASE  = '/content/drive/MyDrive/yolov12n-visdrone/datasets'
DRIVE_ZIP   = os.path.join(DRIVE_BASE, 'VisDrone.zip')
DRIVE_DIR   = os.path.join(DRIVE_BASE, 'VisDrone')

os.makedirs(os.path.join(PROJECT_DIR, 'datasets'), exist_ok=True)

if os.path.exists(YOLO_LOCAL):
    print('[OK] Dataset da ton tai:', YOLO_LOCAL)
elif os.path.exists(DRIVE_ZIP):
    print(f'[INFO] Giai nen {DRIVE_ZIP} ...')
    t0 = time.time()
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
        z.extractall(os.path.join(PROJECT_DIR, 'datasets'))
    print(f'[OK] Giai nen xong! ({time.time()-t0:.0f}s)')
elif os.path.exists(DRIVE_DIR):
    os.symlink(DRIVE_DIR, YOLO_LOCAL)
    print('[OK] Symlink toi', DRIVE_DIR)
else:
    raise FileNotFoundError(
        'Khong tim thay dataset tren Drive!\n'
        f'  Can co: {DRIVE_ZIP}\n'
        f'  Hoac  : {DRIVE_DIR}\n'
        'Hay kiem tra lai thu muc Drive.'
    )

# Kiem tra so luong anh
for split in ['train', 'val', 'test']:
    img_dir = os.path.join(YOLO_LOCAL, 'images', split)
    n = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    status = '[OK]' if n > 0 else '[!!]'
    print(f'  {status} {split:5s}: {n} images')


In [ ]:
!python scripts/check_dataset.py --dataset-root datasets/VisDrone

## 6. Cau hinh training

In [ ]:
import yaml
with open('configs/base.yaml') as f: base = yaml.safe_load(f)
idea_cfg_path = 'configs/idea_cagi.yaml'
try:
    with open(idea_cfg_path) as f: idea = yaml.safe_load(f)
    print('Experiment config:', idea_cfg_path)
    print(yaml.dump(idea, default_flow_style=False))
except FileNotFoundError:
    idea = {}; print('Using base config only')
merged = {**base, **idea}
print('Key params:')
for k in ['epochs','batch','imgsz','lr0','device','test_every_n_epochs']:
    print(f'  {k}: {merged.get(k,"N/A")}')

In [ ]:
IDEA        = 'cagi'
MODEL       = 'yolov12n'
EPOCHS      = 100   # change here
BATCH       = 16    # reduce to 8 if OOM
IMGSZ       = 640
DEVICE      = '0'   # GPU 0
TEST_EVERY  = 10
PROJECT_OUT = 'runs/train'
EXP_NAME    = f'{MODEL}_{IDEA}'
print(f'Model: {MODEL} | Idea: {IDEA} | Epochs: {EPOCHS} | Batch: {BATCH}')
print(f'Output: {PROJECT_OUT}/{EXP_NAME}/')

## 7. Training

- Val tu dong sau moi epoch (ultralytics val=True)
- Test + full metrics report sau moi TEST_EVERY epochs

In [ ]:
import os; os.makedirs('logs', exist_ok=True)
!python train.py \
    --model      {MODEL} \
    --idea       {IDEA} \
    --epochs     {EPOCHS} \
    --batch      {BATCH} \
    --imgsz      {IMGSZ} \
    --device     {DEVICE} \
    --project    {PROJECT_OUT} \
    --name       {EXP_NAME} \
    --test-every {TEST_EVERY} \
    --log-file   logs/{EXP_NAME}.log

### 7b. CAGI -- Phan tich complexity distribution

In [ ]:
import re, os
log = f'logs/{EXP_NAME}.log'
easy, med, hard = [], [], []
pat = r'EASY: ([\d.]+)%.*?MEDIUM: ([\d.]+)%.*?HARD: ([\d.]+)%'
if os.path.exists(log):
    for line in open(log):
        m = re.search(pat, line)
        if m: easy.append(float(m.group(1))); med.append(float(m.group(2))); hard.append(float(m.group(3)))
if easy:
    e,m_,h = sum(easy)/len(easy), sum(med)/len(med), sum(hard)/len(hard)
    ratio  = (e*0.25 + m_*0.5 + h*1.0)/100
    print(f'Avg complexity ({len(easy)} epochs):')
    print(f'  EASY  : {e:.1f}%  (1 block)')
    print(f'  MEDIUM: {m_:.1f}%  (2 blocks)')
    print(f'  HARD  : {h:.1f}%  (4 blocks)')
    print(f'Est. avg FLOPs stage4: {ratio*100:.1f}%')
    print(f'Est. FLOPs saved     : {(1-ratio)*100:.1f}%')
else:
    print('No CAGI log found')

## 8. Danh gia sau training -- Full Metrics Report

3 phan:
1. **Accuracy**: mAP50, mAP50-95, Precision, Recall, AP per-class
2. **Efficiency**: GFLOPs, Params, Model size, Latency, FPS
3. **Efficiency Ratios**: mAP50/GFLOPs, mAP50/Params, mAP50*FPS

In [ ]:
import os
WEIGHTS = f'{PROJECT_OUT}/{EXP_NAME}/weights/best.pt'
if not os.path.exists(WEIGHTS):
    WEIGHTS = f'{PROJECT_OUT}/{EXP_NAME}/weights/last.pt'
    print(f'best.pt not found, using: {WEIGHTS}')
else:
    print(f'Weights: {WEIGHTS}')

In [ ]:
# -- Val split ---------------------------------------------------------
!python val.py \
    --weights        {WEIGHTS} \
    --data           configs/visdrone.yaml \
    --split          val \
    --imgsz          {IMGSZ} \
    --device         {DEVICE} \
    --model-name     {MODEL} \
    --idea           {IDEA} \
    --save-csv       runs/metrics.csv \
    --benchmark-runs 30

In [ ]:
# -- Test split --------------------------------------------------------
!python val.py \
    --weights        {WEIGHTS} \
    --data           configs/visdrone.yaml \
    --split          test \
    --imgsz          {IMGSZ} \
    --device         {DEVICE} \
    --model-name     {MODEL} \
    --idea           {IDEA} \
    --save-csv       runs/metrics.csv \
    --benchmark-runs 30

In [ ]:
import pandas as pd, os
if os.path.exists('runs/metrics.csv'):
    df = pd.read_csv('runs/metrics.csv')
    want = ['timestamp','model','idea','split',
            'acc_map50','acc_map50_95','acc_precision','acc_recall',
            'eff_gflops','eff_params_m','eff_fps','eff_latency_ms',
            'ratio_map50_per_gflop','ratio_map50_x_fps']
    cols = [c for c in want if c in df.columns]
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 200)
    print(df[cols].to_string(index=False))
else:
    print('metrics.csv not found')

## 9. Visualize ket qua training

In [ ]:
from IPython.display import Image, display
import os
exp_dir = f'{PROJECT_OUT}/{EXP_NAME}'
for name in ['results.png','confusion_matrix.png','PR_curve.png','F1_curve.png']:
    p = os.path.join(exp_dir, name)
    if os.path.exists(p):
        print(name); display(Image(p, width=820))

In [ ]:
import pandas as pd, os
rcsv = f'{PROJECT_OUT}/{EXP_NAME}/results.csv'
if os.path.exists(rcsv):
    df = pd.read_csv(rcsv); df.columns = df.columns.str.strip()
    col = 'metrics/mAP50(B)'
    if col in df.columns:
        best = df.loc[df[col].idxmax()]
        print(f'Best epoch : {int(best.get("epoch",0))}')
        print(f'mAP50      : {best[col]:.4f}')
        print(f'mAP50-95   : {best.get("metrics/mAP50-95(B)",0):.4f}')
        print(f'Precision  : {best.get("metrics/precision(B)",0):.4f}')
        print(f'Recall     : {best.get("metrics/recall(B)",0):.4f}')
    else:
        print(df.tail(3).to_string())

## 10. Luu ket qua len Google Drive

In [ ]:
import shutil, os
DRIVE_RUNS = '/content/drive/MyDrive/yolov12n-visdrone/runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)
src = f'{PROJECT_OUT}/{EXP_NAME}'
dst = f'{DRIVE_RUNS}/{EXP_NAME}'
if os.path.exists(src):
    shutil.copytree(src, dst, dirs_exist_ok=True)
    sz = sum(os.path.getsize(os.path.join(d,f))
             for d,_,fs in os.walk(dst) for f in fs)
    print(f'Saved: {dst}  ({sz/1e6:.1f} MB)')
else:
    print('Not found:', src)

In [ ]:
import shutil, os
src = 'runs/metrics.csv'
dst = '/content/drive/MyDrive/yolov12n-visdrone/metrics.csv'
if os.path.exists(src): shutil.copy2(src, dst); print('metrics.csv saved')

In [ ]:
from google.colab import files
import os
pt = f'{PROJECT_OUT}/{EXP_NAME}/weights/best.pt'
if os.path.exists(pt): files.download(pt)
else: print('best.pt not found')

## 11. Demo Inference (tuy chon)

In [ ]:
import glob, random
imgs = glob.glob('datasets/VisDrone/images/test/*.jpg')
if imgs:
    s = random.choice(imgs)
    print('Demo:', s)
    os.system(f'python predict.py --weights {WEIGHTS} --source "{s}"'
              f' --conf 0.25 --imgsz {IMGSZ} --device {DEVICE}'
              f' --project runs/predict --name demo_{IDEA}')

In [ ]:
from IPython.display import Image, display
import glob
for p in glob.glob(f'runs/predict/demo_{IDEA}/*.jpg')[:3]:
    display(Image(p, width=820))